In [ ]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path("/kaggle/input/house-prices-advanced-regression-techniques")

train_path = DATA_DIR / "train.csv"
test_path  = DATA_DIR / "test.csv"

# Load them
train_df = pd.read_csv(train_path)
test_df  = pd.read_csv(test_path)

train_df.head()


In [ ]:
train_df.info()
train_df.describe().T.head(10)
train_df.isnull().sum().sort_values(ascending=False).head(15)

In [ ]:
import numpy as np

none_features = [
    "PoolQC", "Fence", "Alley", "MiscFeature", "FireplaceQu",
    "GarageType", "GarageFinish", "GarageQual", "GarageCond",
    "BsmtQual", "BsmtCond", "BsmtExposure", "BsmtFinType1", "BsmtFinType2",
    "MasVnrType"
]

for col in none_features:
    if col in train_df.columns:
        train_df[col] = train_df[col].fillna("None")

cat_cols = train_df.select_dtypes(exclude=[np.number]).columns
for col in cat_cols:
    if train_df[col].isna().any():
        train_df[col] = train_df[col].fillna("None")

print("\n Missing Values After Cleaning:")
missing_after = train_df.isnull().sum().sort_values(ascending=False)
display(missing_after.head(15))

In [ ]:
import numpy as np
from sklearn.model_selection import KFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression

# Experiment 1:

RANDOM_STATE = 42
TARGET = "SalePrice"

X1 = train_df.drop(columns=[TARGET])
y1 = train_df[TARGET].copy()

num_cols_1 = X1.select_dtypes(include=[np.number]).columns.tolist()
cat_cols_1 = X1.select_dtypes(exclude=[np.number]).columns.tolist()

numeric_pipe_1 = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler(with_mean=False))
])
categorical_pipe_1 = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

pre_1 = ColumnTransformer([
    ("num", numeric_pipe_1, num_cols_1),
    ("cat", categorical_pipe_1, cat_cols_1)
])

model_1 = Pipeline([
    ("pre", pre_1),
    ("reg", LinearRegression())
])

kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scores = cross_val_score(model_1, X1, y1, cv=kf, scoring="neg_mean_squared_error")
rmse_scores = np.sqrt(-scores)

print(f"[Experiment 1] Linear Regression — 5-Fold CV RMSE: {rmse_scores.mean():.2f} ± {rmse_scores.std():.2f}")

In [ ]:
import matplotlib.pyplot as plt

train_df['SalePrice'].hist(bins=50)
plt.title("Distribution of SalePrice")
plt.xlabel("SalePrice ($)")
plt.ylabel("Count")
plt.show()

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge, Lasso, ElasticNet

# Experiment 2:

RANDOM_STATE = 42
TARGET = "SalePrice"

def add_engineered_features(X: pd.DataFrame) -> pd.DataFrame:
    X = X.copy()
    if {"TotalBsmtSF", "1stFlrSF", "2ndFlrSF"}.issubset(X.columns):
        X["TotalSF"] = X["TotalBsmtSF"] + X["1stFlrSF"] + X["2ndFlrSF"]
    if {"YrSold", "YearBuilt"}.issubset(X.columns):
        X["HouseAgeAtSale"] = X["YrSold"] - X["YearBuilt"]
    if {"YearRemodAdd", "YearBuilt"}.issubset(X.columns):
        X["WasRemodeled"] = (X["YearRemodAdd"] > X["YearBuilt"]).astype(int)
    return X

def onehot_dense():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)

X2 = add_engineered_features(train_df.drop(columns=[TARGET]).copy())
y2 = np.log1p(train_df[TARGET].values)

num2 = X2.select_dtypes(include=[np.number]).columns.tolist()
cat2 = X2.select_dtypes(exclude=[np.number]).columns.tolist()

pre2 = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler(with_mean=True))
    ]), num2),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", onehot_dense())
    ]), cat2)
])

ridge = Pipeline([("pre", pre2), ("reg", Ridge(alpha=5.0, random_state=RANDOM_STATE))])
lasso = Pipeline([("pre", pre2), ("reg", Lasso(alpha=0.0005, random_state=RANDOM_STATE, max_iter=10000))])
enet  = Pipeline([("pre", pre2), ("reg", ElasticNet(alpha=0.0005, l1_ratio=0.5, random_state=RANDOM_STATE, max_iter=10000))])

kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
def cv_log_rmse(pipe, X, y):
    scores = cross_val_score(pipe, X, y, cv=kf, scoring="neg_mean_squared_error")
    rmse = np.sqrt(-scores)
    return rmse.mean(), rmse.std()

ridge_rmse, ridge_sd = cv_log_rmse(ridge, X2, y2)
lasso_rmse, lasso_sd = cv_log_rmse(lasso, X2, y2)
enet_rmse,  enet_sd  = cv_log_rmse(enet,  X2, y2)

print(f"[Experiment 2] Ridge (log)     CV RMSE: {ridge_rmse:.4f} ± {ridge_sd:.4f} [log]")
print(f"[Experiment 2] Lasso (log)     CV RMSE: {lasso_rmse:.4f} ± {lasso_sd:.4f} [log]")
print(f"[Experiment 2] ElasticNet(log) CV RMSE: {enet_rmse:.4f} ± {enet_sd:.4f} [log]")

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import ElasticNet

# Experiment 3:

RANDOM_STATE = 42
TARGET = "SalePrice"

def add_engineered_features(X: pd.DataFrame) -> pd.DataFrame:
    X = X.copy()
    if {"TotalBsmtSF", "1stFlrSF", "2ndFlrSF"}.issubset(X.columns):
        X["TotalSF"] = X["TotalBsmtSF"] + X["1stFlrSF"] + X["2ndFlrSF"]
    if {"YrSold", "YearBuilt"}.issubset(X.columns):
        X["HouseAgeAtSale"] = X["YrSold"] - X["YearBuilt"]
    if {"YearRemodAdd", "YearBuilt"}.issubset(X.columns):
        X["WasRemodeled"] = (X["YearRemodAdd"] > X["YearBuilt"]).astype(int)
    return X

def onehot_dense():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)

X3 = add_engineered_features(train_df.drop(columns=[TARGET]).copy())
y3 = np.log1p(train_df[TARGET].values)

if "GrLivArea" in X3.columns:
    cutoff = X3["GrLivArea"].quantile(0.995)
    mask = X3["GrLivArea"] < cutoff
    X3 = X3[mask]
    y3 = y3[mask]

num3 = X3.select_dtypes(include=[np.number]).columns.tolist()
cat3 = X3.select_dtypes(exclude=[np.number]).columns.tolist()

pre3 = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), num3),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", onehot_dense())
    ]), cat3)
])

enet_tuned = Pipeline([
    ("pre", pre3),
    ("reg", ElasticNet(
        alpha=0.0005,
        l1_ratio=0.6,
        max_iter=200000,
        tol=1e-4,
        selection="random",
        random_state=RANDOM_STATE
    ))
])

kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scores = cross_val_score(enet_tuned, X3, y3, cv=kf, scoring="neg_mean_squared_error")
rmse_exp3 = np.sqrt(-scores)

print(f"[Experiment 3] ElasticNet (tuned, log) 5-fold CV RMSE: {rmse_exp3.mean():.4f} ± {rmse_exp3.std():.4f} [log]")
print("Per-fold RMSE (log):", np.round(rmse_exp3, 4))